In [1]:
from stardist.models import StarDist2D
from stardist.plot import render_label
from csbdeep.utils import normalize
import numpy as np
from matplotlib import pyplot as plt
import imageio.v2 as imageio
import os
from tqdm import tqdm
import imageio
# 用 skimage 提取轮廓，在原图上画红色轮廓
from skimage.segmentation import find_boundaries
from scipy.ndimage import binary_dilation

2026-07-28 11:07:23.716619: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-28 11:07:23.797191: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-07-28 11:07:23.797208: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-07-28 11:07:23.814777: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-28 11:07:24.239001: W tensorflow/stream_executor/platform/de

In [2]:
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
print(gpus)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

[]


2026-07-28 11:07:27.221060: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-07-28 11:07:27.222639: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-07-28 11:07:27.224485: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-07-28 11:07:27.226014: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-07-28 11:07:27.227682: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic libra

In [4]:
# StarDist2D.from_pretrained()
# model = StarDist2D.from_pretrained('2D_versatile_he')
# model = StarDist2D(None, "finetuned_he_experiment_xy", "../model/")
model = StarDist2D(None, "finetuned_he_experiment", "../model/")

Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.535431, nms_thresh=0.3.


2026-07-28 11:07:53.558562: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
svs_folder_path = "../data/segmentation_example/my_scan_events/"
# svs_folder_path = "../data/segmentation_example/my_scan_cropped_from_leica_events/"
slice_folders = os.listdir(svs_folder_path)
stardist_output_path = "../results/segmentation_example/"
os.makedirs(stardist_output_path, exist_ok=True)

In [ ]:
for slice_folder in slice_folders:
    os.makedirs(os.path.join(stardist_output_path, slice_folder), exist_ok=True)
    slice_folder_path = os.path.join(svs_folder_path, slice_folder)
    slice_files_x = [f for f in os.listdir(slice_folder_path) if f.endswith("_x.npy")]
    # slice_files_y = [f for f in os.listdir(slice_folder_path) if f.endswith("_y_warp.npy")]
    for slice_file in tqdm(range(len(slice_files_x))):
        slice_file_path_x = os.path.join(slice_folder_path, slice_files_x[slice_file])
        # slice_file_path_y = os.path.join(slice_folder_path, slice_files_y[slice_file])
        slice_x = np.load(slice_file_path_x)
        # slice_y = np.load(slice_file_path_y)
        slice_x = slice_x / np.abs(slice_x).max() / 2 + 0.5
        # slice_y = slice_y / np.abs(slice_y).max() / 2 + 0.5
        # sqrt_slice = np.sqrt(slice_x**2 + slice_y**2)
        # sqrt_slice = sqrt_slice / np.abs(sqrt_slice).max()
        # image = np.stack([slice_x, slice_y, sqrt_slice], axis=-1)
        # # slice = slice[crop_box[0]:crop_box[2], crop_box[1]:crop_box[3]]
        # image = normalize(slice, axis=(0,1))

        image = np.repeat(slice_x[..., np.newaxis], 3, axis=-1)

        # WARNING: If use realworld data, you may need to adjust the scale and prob_thresh parameters for better segmentation results.
        label, _ = model.predict_instances(image, scale=0.5777*0.5777, prob_thresh=0.1)
        # WARNING: else, use the default parameters for synthetic data.
        # label, _ = model.predict_instances(image)

        outlines = find_boundaries(label, mode='inner')
        outlines = binary_dilation(outlines, iterations=2)  # 膨胀轮廓，iterations 越大线越粗

        outline_img = (image * 200 + 55).astype(np.uint8)
        outline_img[outlines] = [255, 0, 0]
        plt.imsave(os.path.join(stardist_output_path, slice_folder, str(slice_file)+"_leica_outline.png"), outline_img)

        # # render_label color overlap with original image
        # rendered_label = render_label(label, img=image, alpha=0.95)
        # print(rendered_label.shape, rendered_label.max(), rendered_label.min())
        # rendered_label = (rendered_label - rendered_label.min()) / (rendered_label.max() - rendered_label.min())
        # background = (label == 0)
        # rendered_label[~background] = [0, 0, 0, 1]
        # rendered_label[background] = [1, 1, 1, 1]
        # plt.imsave(os.path.join(stardist_output_path, slice_folder, slice_file.replace(".npy", "_label_overlap.png")), rendered_label)
        # plt.imsave(os.path.join(stardist_output_path, slice_folder, str(slice_file)+"label_overlap.png"), rendered_label)
        # imageio.imwrite(os.path.join(stardist_output_path, slice_folder, slice_file.replace(".npy", "_label.png")), rendered_label)

100%|██████████| 1/1 [00:05<00:00,  5.16s/it]
